# 🍅 TOMATOES Time Series — Statistical Analysis Notebook
### IMC Prosperity 4 Challenge

This notebook walks through a full statistical characterisation of the TOMATOES price series.  
Each section contains:
- **Theory** — what the test measures and why it matters
- **Code** — clean, reproducible implementation
- **Interpretation guide** — how to read the output

---
**Tests covered:**
1. Data loading & visual overview
2. Stationarity (ADF + KPSS)
3. Drift (OLS trend regression)
4. Autocorrelation (ACF / PACF / Ljung-Box)
5. Cycles (FFT periodogram)
6. Hurst Exponent (R/S analysis)
7. Variance Ratio Test
8. Regression Analysis (trend R² + AR model)
9. Summary table

---
## 0 — Setup & Imports

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Statistical tests ─────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf, q_stat
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from scipy import stats
from scipy.signal import periodogram

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})
TOMATO_RED = '#c0392b'
BLUE       = '#2980b9'

print('All imports OK')

---
## 1 — Load Data & Visual Overview

### 📖 What are we looking at?

Before any test, **always plot your data**. You want to build intuition for:
- The scale and range of prices
- Whether there is a visible trend or mean-reversion
- Any obvious structural breaks or outliers
- How volatile the series is

We also compute **log returns** (`ln(p_t / p_{t-1})`), which is the standard transformation for financial series. Returns are more likely to be stationary and are what we trade on in practice.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LOAD YOUR DATA
# Replace this block with your actual data loading.
# The variable `prices` should be a pd.Series with a numeric index.
# ─────────────────────────────────────────────────────────────────────────────

# Example: df = pd.read_csv('your_file.csv')
#          prices = df['TOMATOES']

# ── PLACEHOLDER: synthetic data for illustration ──────────────────────────────
np.random.seed(42)
n = 1000
# Simulated mean-reverting price around 1000 with slight noise
prices = pd.Series(
    1000 + np.cumsum(np.random.normal(0, 1, n)) * 0.3 +
    5 * np.sin(2 * np.pi * np.arange(n) / 50),   # artificial 50-step cycle
    name='TOMATOES'
)
# ─────────────────────────────────────────────────────────────────────────────

# Derived series
log_prices = np.log(prices)
returns    = log_prices.diff().dropna()

print(f'Series length : {len(prices):,} observations')
print(f'Price range   : [{prices.min():.2f}, {prices.max():.2f}]')
print(f'Mean price    : {prices.mean():.2f}')
print(f'Std dev       : {prices.std():.2f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# ── Raw price ─────────────────────────────────────────────────────────────────
axes[0].plot(prices.values, color=TOMATO_RED, lw=0.9, label='Price')
roll_mean = prices.rolling(50).mean()
axes[0].plot(roll_mean.values, color='black', lw=1.5, ls='--', label='50-step rolling mean')
axes[0].set_title('TOMATOES — Raw Price Series', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend()

# ── Log returns ───────────────────────────────────────────────────────────────
axes[1].plot(returns.values, color=BLUE, lw=0.7, alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Log Returns', fontweight='bold')
axes[1].set_ylabel('log(p_t / p_{t-1})')

# ── Rolling volatility ────────────────────────────────────────────────────────
roll_std = returns.rolling(50).std()
axes[2].fill_between(range(len(roll_std)), roll_std.values, alpha=0.5, color=TOMATO_RED)
axes[2].set_title('Rolling 50-step Volatility (std of returns)', fontweight='bold')
axes[2].set_ylabel('Std Dev')
axes[2].set_xlabel('Time step')

plt.tight_layout()
plt.suptitle('TOMATOES — Overview', y=1.01, fontsize=14, fontweight='bold')
plt.show()

### 🔍 Interpretation guide

| What you see | What it means |
|---|---|
| Rolling mean drifts up/down | Possible non-stationarity or drift |
| Rolling mean stays roughly flat | Mean-stationary |
| Returns cluster into volatile/calm periods | Volatility clustering → ARCH effects |
| Returns look uniformly noisy | Homoskedastic — simpler models may suffice |
| Price bounces around a level | Mean-reversion candidate |

> **Note:** Visual inspection is never conclusive — it primes your expectations before the formal tests below.

---
## 2 — Stationarity: ADF + KPSS Tests

### 📖 Theory

A **stationary** series has:
- Constant mean over time
- Constant variance over time
- Autocovariance that depends only on lag, not on time

Most time-series models (AR, MA, ARIMA) require stationarity — or at least *know* you've violated it.

#### ADF (Augmented Dickey-Fuller)
Tests whether the series has a **unit root**.
- **H₀:** Unit root exists → series is **non-stationary**
- **H₁:** No unit root → series is **stationary**
- Low p-value (< 0.05) → reject H₀ → evidence of stationarity

The ADF regression is: `Δy_t = α + βy_{t-1} + γ₁Δy_{t-1} + ... + ε_t`  
If β = 0, the lagged level has no pull → unit root. If β < 0, there's a mean-reverting force.

#### KPSS (Kwiatkowski–Phillips–Schmidt–Shin)
Tests the **opposite** null:
- **H₀:** Series **is** stationary
- **H₁:** Series has a unit root
- High p-value → fail to reject H₀ → consistent with stationarity

#### Using both together

| ADF p-value | KPSS p-value | Conclusion |
|---|---|---|
| < 0.05 (reject H₀) | > 0.05 (fail to reject H₀) | ✅ Stationary |
| > 0.05 (fail to reject H₀) | < 0.05 (reject H₀) | ❌ Non-stationary |
| < 0.05 | < 0.05 | ⚠️ Trend-stationary |
| > 0.05 | > 0.05 | ⚠️ Insufficient evidence |

In [ ]:
def run_stationarity_tests(series, series_name='Series', verbose=True):
    """
    Runs ADF and KPSS tests and prints a formatted report.
    Returns a dict of results.
    """
    results = {}

    # ── ADF ───────────────────────────────────────────────────────────────────
    # autolag='AIC' automatically picks the number of lagged differences
    # to include so residuals are white noise (avoids over/under-differencing)
    adf_stat, adf_p, adf_lags, adf_nobs, adf_cv, _ = adfuller(series.dropna(), autolag='AIC')
    results['ADF stat']   = adf_stat
    results['ADF p-val']  = adf_p
    results['ADF lags']   = adf_lags

    # ── KPSS ──────────────────────────────────────────────────────────────────
    # regression='c' tests level stationarity
    # regression='ct' tests trend stationarity
    kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(series.dropna(), regression='c', nlags='auto')
    results['KPSS stat']  = kpss_stat
    results['KPSS p-val'] = kpss_p

    if verbose:
        print(f"{'='*55}")
        print(f"  Stationarity Tests: {series_name}")
        print(f"{'='*55}")
        print(f"\n{'─'*30} ADF Test {'─'*15}")
        print(f"  H₀: Unit root exists (non-stationary)")
        print(f"  Test statistic : {adf_stat:.4f}")
        print(f"  p-value        : {adf_p:.4f}")
        print(f"  Lags used      : {adf_lags}")
        for level, cv in adf_cv.items():
            sig = '← REJECT H₀' if adf_stat < cv else ''
            print(f"  Critical value {level}: {cv:.4f}  {sig}")

        print(f"\n{'─'*30} KPSS Test {'─'*14}")
        print(f"  H₀: Series is stationary")
        print(f"  Test statistic : {kpss_stat:.4f}")
        print(f"  p-value        : {kpss_p:.4f}")
        for level, cv in kpss_cv.items():
            sig = '← REJECT H₀' if kpss_stat > cv else ''
            print(f"  Critical value {level}: {cv:.4f}  {sig}")

        print(f"\n{'─'*55}")
        adf_stationary  = adf_p  < 0.05
        kpss_stationary = kpss_p > 0.05
        if adf_stationary and kpss_stationary:
            verdict = '✅ Both tests agree: STATIONARY'
        elif not adf_stationary and not kpss_stationary:
            verdict = '❌ Both tests agree: NON-STATIONARY'
        elif adf_stationary and not kpss_stationary:
            verdict = '⚠️  Conflict — possible TREND-STATIONARY'
        else:
            verdict = '⚠️  Conflict — insufficient evidence'
        print(f"  Verdict: {verdict}")
        print(f"{'='*55}\n")

    return results


# Run on price levels and on returns
print("\n" + "#"*60)
print("  PRICE LEVELS")
print("#"*60)
res_price = run_stationarity_tests(prices, 'Price levels')

print("\n" + "#"*60)
print("  LOG RETURNS")
print("#"*60)
res_returns = run_stationarity_tests(returns, 'Log returns')

### 🔍 Interpretation guide

**If price levels are non-stationary but returns are stationary** → the price series is **I(1)** (integrated of order 1), which is the standard result for financial prices. This means:
- You should model and analyse **returns**, not raw prices
- Mean-reversion strategies should work on *deviations from a spread or moving average*, not on the raw price

**If price levels are stationary** → prices gravitate to a long-run mean. Mean-reversion strategies on the raw price may be viable.

**Trend-stationary** → subtract a fitted linear trend from the price, then re-test. The detrended series may be stationary.

---
## 3 — Drift: OLS Trend Regression

### 📖 Theory

**Drift** is a systematic directional bias — the series tends to move up (or down) on average each step. It is distinct from a random walk, where direction at each step is unpredictable.

We test drift by fitting:  
$$p_t = \alpha + \beta \cdot t + \varepsilon_t$$

- **β** is the drift coefficient: the expected change in price per time step
- **t-statistic on β**: if |t| > 2 and p < 0.05 → drift is statistically significant
- **R²**: how much price variance is explained by the linear trend alone

We also inspect **rolling mean behaviour** — a drifting series will have a rolling mean that systematically moves in one direction.

In [ ]:
# ── OLS regression: price ~ constant + time ───────────────────────────────────
t = np.arange(len(prices))
X = add_constant(t)         # adds intercept column
y = prices.values

ols_model  = OLS(y, X).fit()
alpha_hat  = ols_model.params[0]
beta_hat   = ols_model.params[1]
beta_tstat = ols_model.tvalues[1]
beta_pval  = ols_model.pvalues[1]
r_squared  = ols_model.rsquared

print("=" * 50)
print("  Drift Test: OLS  price ~ α + β·t")
print("=" * 50)
print(f"  Intercept α      : {alpha_hat:.4f}")
print(f"  Slope β (drift)  : {beta_hat:.6f} per step")
print(f"  t-statistic (β)  : {beta_tstat:.4f}")
print(f"  p-value (β)      : {beta_pval:.4f}")
print(f"  R²               : {r_squared:.4f}")
print()
if abs(beta_tstat) > 2 and beta_pval < 0.05:
    direction = 'upward' if beta_hat > 0 else 'downward'
    print(f"  → Statistically significant {direction} drift detected.")
    print(f"    Expected {direction} move of {abs(beta_hat):.6f} per step")
else:
    print("  → No statistically significant drift detected.")
print("=" * 50)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── Left: price + fitted trend line ───────────────────────────────────────────
fitted = ols_model.fittedvalues
axes[0].plot(prices.values, color=TOMATO_RED, lw=0.8, alpha=0.8, label='Price')
axes[0].plot(fitted, color='black', lw=2, ls='--', label=f'OLS trend (β={beta_hat:.4f})')
axes[0].set_title('Price vs. Fitted Linear Trend', fontweight='bold')
axes[0].set_xlabel('Time step')
axes[0].legend()

# ── Right: OLS residuals ───────────────────────────────────────────────────────
residuals = ols_model.resid
axes[1].plot(residuals, color=BLUE, lw=0.7, alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Detrended Residuals (price - trend)', fontweight='bold')
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

### 🔍 Interpretation guide

| Result | Meaning |
|---|---|
| Significant β, high R² | Strong deterministic trend — prices follow a predictable slope |
| Significant β, low R² | Weak drift overwhelmed by noise |
| Non-significant β | No evidence of drift — price has no long-run directional bias |
| Residuals look stationary | After removing the trend, the series may be stationary (trend-stationary process) |
| Residuals still wander | Even after detrending, stochastic non-stationarity remains |

> **Trading implication:** Significant drift in the *competition window* may be exploitable by a directional bias, but drift can be short-lived. Combine with stationarity tests.

---
## 4 — Autocorrelation: ACF, PACF & Ljung-Box Test

### 📖 Theory

**Autocorrelation at lag k** measures the linear correlation between `y_t` and `y_{t-k}`.
$$\rho_k = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}$$

In an **efficient market**, returns should have zero autocorrelation — past prices contain no information about future prices.

#### ACF (Autocorrelation Function)
Plots ρ_k for all lags k. Includes the *indirect* effect of intermediate lags.

#### PACF (Partial ACF)
Plots the correlation at lag k *after removing* the effect of lags 1 through k-1.  
Useful for identifying the order of an AR process:
- If PACF cuts off at lag p → AR(p) process

#### Ljung-Box Test
Formal joint test: are *all* autocorrelations up to lag k simultaneously zero?
- **H₀:** No autocorrelation up to lag k
- Low p-value → significant autocorrelation present

**Confidence bands** in ACF/PACF plots are ±1.96/√N — spikes outside these are significant at the 5% level.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

max_lags = 60

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# ── ACF of price levels ────────────────────────────────────────────────────────
plot_acf(prices,   lags=max_lags, ax=axes[0, 0], color=TOMATO_RED,
         title='ACF — Price Levels', alpha=0.05)

# ── PACF of price levels ──────────────────────────────────────────────────────
plot_pacf(prices,  lags=max_lags, ax=axes[0, 1], color=TOMATO_RED,
          title='PACF — Price Levels', method='ywm', alpha=0.05)

# ── ACF of returns ─────────────────────────────────────────────────────────────
plot_acf(returns,  lags=max_lags, ax=axes[1, 0], color=BLUE,
         title='ACF — Log Returns', alpha=0.05)

# ── PACF of returns ────────────────────────────────────────────────────────────
plot_pacf(returns, lags=max_lags, ax=axes[1, 1], color=BLUE,
          title='PACF — Log Returns', method='ywm', alpha=0.05)

for ax in axes.flat:
    ax.set_xlabel('Lag')

plt.tight_layout()
plt.show()

In [ ]:
# ── Ljung-Box formal test on returns ──────────────────────────────────────────
# We test at several lag horizons to see how autocorrelation accumulates
lags_to_test = [5, 10, 20, 30, 50]
lb_results   = acorr_ljungbox(returns.dropna(), lags=lags_to_test, return_df=True)

print("=" * 55)
print("  Ljung-Box Test on Log Returns")
print("  H₀: No autocorrelation up to lag k")
print("=" * 55)
print(f"  {'Lag':>5}  {'LB Statistic':>14}  {'p-value':>10}  {'Significant?':>14}")
print("  " + "─"*50)
for lag, row in lb_results.iterrows():
    sig = '✅ YES' if row['lb_pvalue'] < 0.05 else '  no'
    print(f"  {lag:>5}  {row['lb_stat']:>14.4f}  {row['lb_pvalue']:>10.4f}  {sig:>14}")
print("=" * 55)

In [ ]:
# ── Autocorrelation magnitude at each lag ─────────────────────────────────────
acf_values  = acf(returns.dropna(),  nlags=max_lags)[1:]   # skip lag 0 (=1)
conf_band   = 1.96 / np.sqrt(len(returns))

fig, ax = plt.subplots(figsize=(13, 4))
lags_range = np.arange(1, max_lags + 1)
colors = [TOMATO_RED if abs(v) > conf_band else BLUE for v in acf_values]
ax.bar(lags_range, acf_values, color=colors, alpha=0.75, label='ACF')
ax.axhline( conf_band, ls='--', color='grey', lw=1.2, label='95% confidence band')
ax.axhline(-conf_band, ls='--', color='grey', lw=1.2)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('ACF of Log Returns — Significant lags in red', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

sig_lags = lags_range[np.abs(acf_values) > conf_band]
print(f"Significant autocorrelation at lags: {sig_lags.tolist()}")

### 🔍 Interpretation guide

**ACF pattern → Process type:**

| ACF shape | PACF shape | Likely model |
|---|---|---|
| Decays slowly (geometrically) | Sharp cutoff at lag p | AR(p) |
| Sharp cutoff at lag q | Decays slowly | MA(q) |
| Both decay slowly | Both decay slowly | ARMA(p,q) |
| All near zero | All near zero | White noise (efficient) |
| Decays very slowly (hyperbolic) | Same | Long memory process |
| Oscillates with period k | Peaks at multiples of k | Cyclical / seasonal |

**Ljung-Box:** If significant at short lags (5–10), the autocorrelation is economically relevant. Significant only at long lags might be noise or model mis-specification.

---
## 5 — Cycles: FFT Periodogram

### 📖 Theory

**Fourier analysis** decomposes a time series into a sum of sine and cosine waves at different frequencies. If a particular frequency dominates, the series has a **periodic cycle** at that wavelength.

The **periodogram** plots power (squared amplitude) against frequency:
- A peak at frequency f → a cycle with period T = N/f (or 1/f if frequency is in cycles-per-step)
- Flat periodogram → no dominant cycle (consistent with white noise)

We apply the FFT to:
1. **Detrended prices** — to find structural price cycles
2. **Returns** — to find return predictability cycles

> **Important:** We detrend before FFT. A strong trend injects power at all low frequencies and masks real cycles.

In [ ]:
def plot_periodogram(series, series_name='Series', top_n=5):
    """
    Computes and plots the FFT periodogram, labels the top-n dominant periods.
    """
    s = series.dropna().values
    N = len(s)

    # Detrend by subtracting linear fit
    t      = np.arange(N)
    coeffs = np.polyfit(t, s, 1)
    detrended = s - np.polyval(coeffs, t)

    # FFT
    fft_vals = np.fft.rfft(detrended)
    freqs    = np.fft.rfftfreq(N)     # cycles per step
    power    = np.abs(fft_vals) ** 2

    # Skip frequency 0 (DC component)
    freqs = freqs[1:]
    power = power[1:]
    periods = 1.0 / freqs            # period in steps

    # Top-n dominant periods
    top_idx     = np.argsort(power)[::-1][:top_n]
    top_periods = periods[top_idx]
    top_power   = power[top_idx]

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Power vs frequency
    axes[0].plot(freqs, power, color=TOMATO_RED, lw=0.8)
    for idx in top_idx:
        axes[0].axvline(freqs[idx], color='black', ls='--', lw=0.8, alpha=0.6)
        axes[0].text(freqs[idx], power[idx], f' T≈{periods[idx]:.1f}',
                     fontsize=8, va='bottom')
    axes[0].set_xlabel('Frequency (cycles per step)')
    axes[0].set_ylabel('Power')
    axes[0].set_title(f'Periodogram — {series_name}', fontweight='bold')

    # Power vs period (more interpretable)
    # Only show periods < N/2 (Nyquist)
    mask = periods <= N // 2
    axes[1].plot(periods[mask], power[mask], color=BLUE, lw=0.8)
    for idx in top_idx:
        if periods[idx] <= N // 2:
            axes[1].axvline(periods[idx], color='black', ls='--', lw=0.8, alpha=0.6)
            axes[1].text(periods[idx], power[idx], f' T≈{periods[idx]:.1f}',
                         fontsize=8, va='bottom')
    axes[1].set_xlabel('Period (steps per cycle)')
    axes[1].set_ylabel('Power')
    axes[1].set_title(f'Power vs Period — {series_name}', fontweight='bold')
    axes[1].set_xlim([2, N // 2])

    plt.tight_layout()
    plt.show()

    print(f"\nTop {top_n} dominant cycles in {series_name}:")
    print(f"  {'Rank':>4}  {'Period (steps)':>16}  {'Power':>12}  {'Freq':>10}")
    print("  " + "─"*48)
    for rank, (p, pw, f) in enumerate(zip(top_periods, top_power, freqs[top_idx]), 1):
        print(f"  {rank:>4}  {p:>16.2f}  {pw:>12.2f}  {f:>10.6f}")
    print()


plot_periodogram(prices,  series_name='Price Levels (detrended)')
plot_periodogram(returns, series_name='Log Returns')

### 🔍 Interpretation guide

| Periodogram shape | Meaning |
|---|---|
| Flat, no dominant peak | No cyclical structure — consistent with white noise |
| One dominant sharp peak at period T | Clear cycle of length T steps |
| Multiple peaks at T, T/2, T/3 | Harmonic structure (strong fundamental cycle) |
| Power concentrated at low frequencies | Long, slow cycles / trend-like behaviour |
| Power decays as power law in frequency | Possible long-memory or fractal structure |

> **Competition implication:** A cycle of period T in returns means that on average, every T steps the price completes one mean-reversion cycle. This directly informs entry/exit timing — enter near cycle troughs, exit near peaks.
>
> **Caveat:** FFT peaks must be robust — always check if the same cycle appears in different sub-windows of the data before trading on it.

---
## 6 — Hurst Exponent: R/S Analysis

### 📖 Theory

The **Hurst Exponent H** characterises the *memory* and *self-similarity* of a time series.

| H | Process type | Behaviour |
|---|---|---|
| H = 0.5 | Random walk | No memory — efficient market |
| H > 0.5 | Persistent / trending | Trends persist — momentum strategies |
| H < 0.5 | Anti-persistent / mean-reverting | Reversals are more likely — mean-reversion strategies |

#### R/S (Rescaled Range) Method

For a window of size n:
1. Compute cumulative deviations from the mean: `Z_t = Σ(r_i - r̄)`
2. Range: `R = max(Z) - min(Z)`
3. Std dev: `S = std(r)`
4. Rescaled range: `R/S`

Repeat for many window sizes n. For a fractal process: `E[R/S] ~ c · n^H`

Taking logs: `log(R/S) = log(c) + H · log(n)`

So **H is the slope** of log(R/S) vs log(n) — we estimate it via linear regression.

In [ ]:
def compute_hurst_rs(series, min_window=10, max_window=None, num_points=30):
    """
    Estimates the Hurst exponent using R/S analysis.
    Returns H, the log(n) values, log(R/S) values, and regression details.
    """
    s = np.array(series.dropna())
    N = len(s)
    if max_window is None:
        max_window = N // 2

    # Logarithmically spaced window sizes
    window_sizes = np.unique(
        np.logspace(np.log10(min_window), np.log10(max_window), num_points).astype(int)
    )

    rs_vals = []

    for n in window_sizes:
        # Split series into non-overlapping windows of size n
        n_chunks = N // n
        if n_chunks < 1:
            continue
        rs_chunk = []
        for i in range(n_chunks):
            chunk = s[i*n : (i+1)*n]
            mean  = chunk.mean()
            # Cumulative deviations from the mean
            cum_dev = np.cumsum(chunk - mean)
            R = cum_dev.max() - cum_dev.min()   # Range
            S = chunk.std(ddof=1)               # Std dev
            if S > 0:
                rs_chunk.append(R / S)
        if rs_chunk:
            rs_vals.append(np.mean(rs_chunk))
        else:
            window_sizes = window_sizes[window_sizes != n]

    log_n  = np.log(window_sizes[:len(rs_vals)])
    log_rs = np.log(rs_vals)

    # Linear regression: log(R/S) = a + H * log(n)
    slope, intercept, r_val, p_val, std_err = stats.linregress(log_n, log_rs)
    H = slope

    return H, log_n, log_rs, intercept, r_val**2, std_err


H, log_n, log_rs, intercept, r2, std_err = compute_hurst_rs(prices)

print("=" * 50)
print("  Hurst Exponent — R/S Analysis")
print("=" * 50)
print(f"  H estimate  : {H:.4f}")
print(f"  Std error   : {std_err:.4f}")
print(f"  95% CI      : [{H - 1.96*std_err:.4f}, {H + 1.96*std_err:.4f}]")
print(f"  R² of fit   : {r2:.4f}")
print()
if H > 0.55:
    verdict = f'📈 TRENDING / PERSISTENT (H={H:.3f} > 0.5)'
elif H < 0.45:
    verdict = f'↩️  MEAN-REVERTING / ANTI-PERSISTENT (H={H:.3f} < 0.5)'
else:
    verdict = f'🎲 RANDOM WALK-LIKE (H={H:.3f} ≈ 0.5)'
print(f"  Verdict: {verdict}")
print("=" * 50)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(log_n, log_rs, color=TOMATO_RED, s=40, zorder=5, label='R/S data points')
ax.plot(log_n, intercept + H * log_n, color='black', lw=2,
        label=f'OLS fit: slope H = {H:.4f}')

# Reference lines for H=0.5 (random walk)
ax.plot(log_n, intercept + 0.5 * log_n, color=BLUE, lw=1.5, ls='--',
        label='H = 0.5 (random walk reference)')

ax.set_xlabel('log(window size n)')
ax.set_ylabel('log(R/S)')
ax.set_title(f'Hurst R/S Analysis  —  H = {H:.4f}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 🔍 Interpretation guide

- **The log-log plot should be approximately linear.** If it curves, the Hurst exponent is not constant across scales — the process may have regime changes.
- **Compare the fitted slope to the H=0.5 dashed line.** Visually check whether your data clearly departs from the random-walk reference.
- **R² of the regression:** Should be high (> 0.95) for a reliable estimate. Low R² means the R/S scaling is not clean — be skeptical of H.
- **Confidence interval:** If the interval contains 0.5, you cannot statistically distinguish the series from a random walk.

> **Practical note:** R/S analysis is sensitive to short-range autocorrelation. For a more robust estimate, consider also using the `hurst` library which implements DFA (Detrended Fluctuation Analysis).

---
## 7 — Variance Ratio Test

### 📖 Theory

The **Variance Ratio (VR) test** (Lo & MacKinlay, 1988) tests whether a series is consistent with a random walk.

**Key insight:** For a random walk, variance grows *linearly* with time:
$$\text{Var}(r_t + r_{t+1} + ... + r_{t+k-1}) = k \cdot \text{Var}(r_t)$$

So the **variance ratio** is defined as:
$$\text{VR}(k) = \frac{\text{Var}(k\text{-period returns})}{k \cdot \text{Var}(1\text{-period returns})}$$

Under a random walk, VR(k) = 1 for all k.

| VR(k) > 1 | Positive autocorrelation at horizon k → **momentum** |
|---|---|
| VR(k) < 1 | Negative autocorrelation at horizon k → **mean-reversion** |
| VR(k) = 1 | Random walk — no exploitable structure |

Testing at *multiple horizons* k gives a richer picture: the series might be mean-reverting at short horizons but trending at long horizons.

In [ ]:
def variance_ratio(series, k):
    """
    Computes the variance ratio VR(k) and its z-statistic.
    Uses the heteroskedasticity-robust version (Lo-MacKinlay).

    Returns: VR estimate, z-statistic, p-value
    """
    r  = np.array(series.dropna())   # 1-period returns
    n  = len(r)
    mu = r.mean()

    # Variance of 1-period returns
    sigma2_1 = np.sum((r - mu)**2) / (n - 1)

    # k-period returns (overlapping)
    r_k = np.array([np.sum(r[t:t+k]) for t in range(n - k + 1)])

    # Variance of k-period returns (unbiased)
    m        = k * (n - k + 1) * (1 - k / n)
    sigma2_k = np.sum((r_k - k * mu)**2) / m

    VR = sigma2_k / (k * sigma2_1)

    # Heteroskedasticity-robust standard error (Lo-MacKinlay)
    delta = np.array([
        np.sum(
            (r[j:]  - mu)**2 * (r[j-lag:-lag] - mu)**2
            for j in range(lag, n)
        ).sum() / (np.sum((r - mu)**2))**2
        for lag in range(1, k)
    ])
    theta = np.sum([(1 - lag/k)**2 * delta[lag-1] for lag in range(1, k)])

    z_stat = (VR - 1) / np.sqrt(theta)
    p_val  = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    return VR, z_stat, p_val


# Test at multiple horizons
horizons = [2, 4, 8, 16, 32]

print("=" * 65)
print("  Variance Ratio Test on Log Returns")
print("  H₀: VR(k) = 1  (random walk at horizon k)")
print("=" * 65)
print(f"  {'k':>4}  {'VR(k)':>8}  {'z-stat':>10}  {'p-value':>10}  {'Interpretation':>20}")
print("  " + "─"*60)

vr_values = []
for k in horizons:
    try:
        vr, z, p = variance_ratio(returns, k)
    except Exception:
        vr, z, p = np.nan, np.nan, np.nan
    vr_values.append(vr)
    if p < 0.05:
        interp = 'Momentum ↑' if vr > 1 else 'Mean-rev ↓'
    else:
        interp = 'Random walk'
    print(f"  {k:>4}  {vr:>8.4f}  {z:>10.4f}  {p:>10.4f}  {interp:>20}")

print("=" * 65)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(horizons, vr_values, 'o-', color=TOMATO_RED, lw=2, ms=8, label='VR(k)')
ax.axhline(1.0, color='black', lw=1.5, ls='--', label='VR = 1 (random walk)')
ax.fill_between(horizons, 0.95, 1.05, alpha=0.1, color='grey', label='±5% band')

for k, vr in zip(horizons, vr_values):
    ax.annotate(f'{vr:.3f}', (k, vr), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)

ax.set_xlabel('Horizon k (steps)')
ax.set_ylabel('Variance Ratio VR(k)')
ax.set_title('Variance Ratio Test — Deviation from Random Walk', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 🔍 Interpretation guide

- **VR profile consistently below 1:** Mean-reverting at all tested horizons → pairs/stat-arb strategies, sell rallies, buy dips.
- **VR profile consistently above 1:** Trending at all horizons → momentum strategies.
- **VR < 1 at short k, VR > 1 at long k:** Short-term mean-reversion, long-term trend — typical of real asset prices.
- **VR ≈ 1 at all k:** Cannot reject the random walk — exploit only with conviction from other tests.

> **Combine with Hurst:** The Hurst exponent gives one number; the VR profile shows *at which time horizons* the effect is strongest — crucial for sizing your trade holding period.

---
## 8 — Regression Analysis: Trend R² & AR Model

### 📖 Theory

Two distinct regression analyses:

#### A — Trend regression (already done in §3)
$$p_t = \alpha + \beta t + \varepsilon_t$$
- R² measures how much price movement is a pure function of time (deterministic drift)
- t-stat on β tests whether the drift is real

#### B — Autoregressive model: AR(p)
$$r_t = \phi_0 + \phi_1 r_{t-1} + \phi_2 r_{t-2} + ... + \phi_p r_{t-p} + \varepsilon_t$$
- R² measures how much of *today's return* is explained by *past returns*
- High R² → strong predictability from history → exploitable
- t-stats on each φ_i tell you which specific lags drive the predictability
- **Residual diagnostics:** If the AR model is well-specified, residuals should be white noise (no remaining autocorrelation)

**Choosing p:** Use AIC/BIC to select the optimal lag order — lower is better.

In [ ]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.stats.stattools import durbin_watson

# ── A: Trend regression summary ───────────────────────────────────────────────
print("=" * 55)
print("  A — Trend Regression: price ~ α + β·t")
print("=" * 55)
print(f"  R²           : {r_squared:.4f}")
print(f"  β (drift)    : {beta_hat:.6f}")
print(f"  t-stat (β)   : {beta_tstat:.4f}")
print(f"  p-value (β)  : {beta_pval:.4f}")
print()

# ── B: Select best AR lag order via AIC ───────────────────────────────────────
max_ar_lags = 20
aic_scores  = {}
bic_scores  = {}

for p in range(1, max_ar_lags + 1):
    try:
        m = AutoReg(returns.dropna(), lags=p, old_names=False).fit()
        aic_scores[p] = m.aic
        bic_scores[p] = m.bic
    except Exception:
        pass

best_p_aic = min(aic_scores, key=aic_scores.get)
best_p_bic = min(bic_scores, key=bic_scores.get)
print(f"  Best AR lag (AIC): p = {best_p_aic}  (AIC = {aic_scores[best_p_aic]:.2f})")
print(f"  Best AR lag (BIC): p = {best_p_bic}  (BIC = {bic_scores[best_p_bic]:.2f})")
print()

In [ ]:
# ── Fit the best AR(p) model ──────────────────────────────────────────────────
best_p = best_p_aic   # use AIC selection
ar_model = AutoReg(returns.dropna(), lags=best_p, old_names=False).fit()
ar_resid = ar_model.resid

print("=" * 55)
print(f"  B — AR({best_p}) Model on Log Returns")
print("=" * 55)
print(f"  R²                : {ar_model.rsquared:.4f}")
print(f"  AIC               : {ar_model.aic:.4f}")
print(f"  BIC               : {ar_model.bic:.4f}")
print(f"  Durbin-Watson     : {durbin_watson(ar_resid):.4f}")
print()
print("  Coefficients:")
print(f"  {'Lag':>6}  {'Coeff':>10}  {'t-stat':>10}  {'p-value':>10}  {'Sig':>5}")
print("  " + "─"*48)
for name, coef, tstat, pval in zip(
    ar_model.params.index,
    ar_model.params,
    ar_model.tvalues,
    ar_model.pvalues
):
    sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else ''))
    print(f"  {name:>6}  {coef:>10.6f}  {tstat:>10.4f}  {pval:>10.4f}  {sig:>5}")
print("=" * 55)

In [ ]:
# ── AIC/BIC across lag orders ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

lags_list = list(aic_scores.keys())
axes[0].plot(lags_list, [aic_scores[p] for p in lags_list], 'o-', color=TOMATO_RED)
axes[0].axvline(best_p_aic, color='black', ls='--', lw=1.5, label=f'Best p={best_p_aic}')
axes[0].set_title('AIC vs AR Lag Order', fontweight='bold')
axes[0].set_xlabel('Lag p')
axes[0].set_ylabel('AIC')
axes[0].legend()

# ── AR residuals ──────────────────────────────────────────────────────────────
axes[1].plot(ar_resid.values, color=BLUE, lw=0.7, alpha=0.8)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title(f'AR({best_p}) Residuals', fontweight='bold')
axes[1].set_xlabel('Time step')

# ── Residual ACF ──────────────────────────────────────────────────────────────
plot_acf(ar_resid.dropna(), lags=40, ax=axes[2], color=TOMATO_RED,
         title=f'ACF of AR({best_p}) Residuals', alpha=0.05)
axes[2].set_xlabel('Lag')

plt.suptitle('AR Model Diagnostics', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Ljung-Box on residuals ────────────────────────────────────────────────────
lb_resid = acorr_ljungbox(ar_resid.dropna(), lags=[10, 20], return_df=True)
print("\nLjung-Box on AR residuals (should be non-significant if model is well-specified):")
print(lb_resid.to_string())

### 🔍 Interpretation guide

**AR model R²:**
- R² = 0.01–0.05: Weak but potentially exploitable predictability (financial data rarely exceeds this)
- R² = 0.05–0.20: Moderate predictability — meaningful signal
- R² > 0.20: Strong predictability — review carefully for data leakage

**Significant coefficients (φ_i):**
- Positive φ at lag 1 → positive serial correlation → last return predicts continuation
- Negative φ at lag 1 → negative serial correlation → last return predicts reversal

**Durbin-Watson statistic:**
- DW ≈ 2.0: No first-order autocorrelation in residuals (good)
- DW < 1.5: Positive autocorrelation in residuals → model is under-specified, add more lags
- DW > 2.5: Negative autocorrelation in residuals

**Residual ACF:**
- All spikes within confidence bands → model has captured the autocorrelation structure → white noise residuals → good fit
- Remaining significant spikes → model under-specified → increase p or consider a different model class

---
## 9 — Summary Table

A consolidated view of all test results for quick reference and trading implication.

In [ ]:
# ── Collect all results into a summary ────────────────────────────────────────

# Re-run stationarity quietly to get values
res_p = run_stationarity_tests(prices,  verbose=False)
res_r = run_stationarity_tests(returns, verbose=False)

# Variance ratio at k=2 (shortest horizon)
try:
    vr2, vr2_z, vr2_p = variance_ratio(returns, 2)
except Exception:
    vr2, vr2_z, vr2_p = np.nan, np.nan, np.nan

rows = [
    ('Stationarity',   'ADF (prices)',         f"p={res_p['ADF p-val']:.3f}",
     '✅ Stationary' if res_p['ADF p-val'] < 0.05 else '❌ Non-stationary'),

    ('Stationarity',   'KPSS (prices)',        f"p={res_p['KPSS p-val']:.3f}",
     '✅ Consistent' if res_p['KPSS p-val'] > 0.05 else '❌ Non-stationary'),

    ('Stationarity',   'ADF (returns)',        f"p={res_r['ADF p-val']:.3f}",
     '✅ Stationary' if res_r['ADF p-val'] < 0.05 else '❌ Non-stationary'),

    ('Drift',          'OLS β (prices)',       f"t={beta_tstat:.3f}, p={beta_pval:.3f}",
     '⚠️  Drift present' if beta_pval < 0.05 else '✅ No drift'),

    ('Drift',          'OLS R² (prices)',      f"{r_squared:.4f}",
     'High trend' if r_squared > 0.5 else 'Low trend'),

    ('Autocorrelation','Ljung-Box (lag 10)',
     f"p={lb_results['lb_pvalue'].iloc[1]:.3f}",
     '⚠️  Autocorr.' if lb_results['lb_pvalue'].iloc[1] < 0.05 else '✅ No autocorr.'),

    ('Hurst',          'R/S estimate',         f"H={H:.4f} ± {std_err:.4f}",
     '📈 Trending' if H > 0.55 else ('↩️  Mean-rev.' if H < 0.45 else '🎲 Random walk')),

    ('Variance Ratio', 'VR(k=2)',              f"VR={vr2:.4f}, p={vr2_p:.3f}",
     '📈 Momentum' if (vr2 > 1 and vr2_p < 0.05)
     else ('↩️  Mean-rev.' if (vr2 < 1 and vr2_p < 0.05) else '🎲 Random walk')),

    ('AR Regression',  f'AR({best_p}) R²',    f"{ar_model.rsquared:.4f}",
     'Predictable' if ar_model.rsquared > 0.02 else 'Weak signal'),

    ('AR Regression',  'Residual LB (lag 10)',
     f"p={lb_resid['lb_pvalue'].iloc[0]:.3f}",
     '✅ Good fit' if lb_resid['lb_pvalue'].iloc[0] > 0.05 else '⚠️  Mis-specified'),
]

summary_df = pd.DataFrame(rows, columns=['Category', 'Test', 'Statistic', 'Conclusion'])

# Style and display
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 120)
print("\n" + "═"*80)
print("  TOMATOES — FULL STATISTICAL ANALYSIS SUMMARY")
print("═"*80)
print(summary_df.to_string(index=False))
print("═"*80)

In [ ]:
# ── Visual summary heatmap ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')

col_labels = ['Category', 'Test', 'Statistic', 'Conclusion']
table_data  = [list(row) for row in rows]

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='left'
)
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1, 1.7)

# Header styling
for j in range(len(col_labels)):
    table[(0, j)].set_facecolor('#2c3e50')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Row zebra-striping
for i in range(1, len(rows) + 1):
    bg = '#fdedec' if i % 2 == 0 else '#fdfefe'
    for j in range(len(col_labels)):
        table[(i, j)].set_facecolor(bg)

ax.set_title('TOMATOES — Statistical Analysis Summary', fontsize=13,
             fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---
## 10 — Trading Strategy Implications

Use this cell to synthesise the results into a strategy view.

| Finding | Strategy implication |
|---|---|
| Price non-stationary, returns stationary | Trade returns, not price levels; use z-score on spread/MA deviation |
| Significant drift | Add a directional bias; adjust entry thresholds asymmetrically |
| Significant autocorrelation at lag 1 | AR(1) signal: positive → momentum; negative → fade the last move |
| Dominant FFT cycle at period T | Use T as holding period; buy at cycle trough, sell at peak |
| H < 0.5 | Mean-reversion is the primary edge; use Bollinger Bands or Ornstein-Uhlenbeck models |
| H > 0.5 | Trend-following edge; use moving-average crossovers or momentum signals |
| VR < 1 at k=2 | Short-horizon mean-reversion; tight bands |
| AR(p) R² > 0.02 | Quantitative signal: forecast next return from AR coefficients; bet proportionally |

---
*Notebook complete. Replace the placeholder data with your actual TOMATOES series to get real results.*